In [1]:
import logging

logger = logging.getLogger("api")

logger.setLevel(logging.DEBUG)

formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)s | %(message)s"
)

file_handler = logging.FileHandler("api.log")
file_handler.setFormatter(formatter)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [2]:
import time
from functools import wraps

def timing(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        duration = end - start

        logger.info(f"{func.__name__} executed in {duration:.4f} seconds")

        return result

    return wrapper

In [3]:
def log_io(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        logger.debug(f"{func.__name__} called with args={args}, kwargs={kwargs}")

        result = func(*args, **kwargs)

        logger.debug(f"{func.__name__} returned {result}")

        return result

    return wrapper

In [4]:
def require_role(role):

    def decorator(func):

        @wraps(func)
        def wrapper(user_role, *args, **kwargs):

            if user_role != role:
                logger.warning(
                    f"Access denied for role={user_role} on {func.__name__}"
                )
                raise PermissionError("Unauthorized")

            logger.info(f"Access granted for role={user_role}")

            return func(user_role, *args, **kwargs)

        return wrapper

    return decorator

In [5]:
@timing
@log_io
@require_role("admin")
def delete_user(user_role, user_id):
    return f"User {user_id} deleted"

In [6]:
delete_user("admin", 42)

2026-03-08 04:56:30,401 | api | DEBUG | delete_user called with args=('admin', 42), kwargs={}
2026-03-08 04:56:30,404 | api | INFO | Access granted for role=admin
2026-03-08 04:56:30,405 | api | DEBUG | delete_user returned User 42 deleted
2026-03-08 04:56:30,406 | api | INFO | delete_user executed in 0.0047 seconds


'User 42 deleted'

In [7]:
delete_user("guest", 42)

2026-03-08 04:56:48,482 | api | DEBUG | delete_user called with args=('guest', 42), kwargs={}
2026-03-08 04:56:48,485 | api | WARNING | Access denied for role=guest on delete_user


PermissionError: Unauthorized